# WP4 - Morphology-aware UrbanFormer-Field (a pre-registered null result)

**Question.** Does a global morphology vector
`m = [lambda_p, lambda_f, h_m, h_rms, h_skew, h_kurt, gamma_m, h_max]` add
predictive power on top of the building tokens? The single isolated lever is
`MORPH_MODE in {"none", "token", "query"}`; everything else (encoder, relative
cross-attention, axial self-attention, depth, loss, augmentation) is frozen at
the UF-F config, so any R2 change is attributable to `m` alone.

**Headline (null result, as pre-registered).**

| MORPH_MODE | how m enters | R2 (core test) |
|---|---|---|
| `none` (= WP3 UF-F control) | m unused | **0.8461** |
| `token` | m -> MLP -> global token, prepended to the building set | 0.8358 |

`R2(token) ~ R2(none)`, and the `token` variant is if anything slightly worse. The
morphology vector adds nothing because **the building tokens already encode `m`**:
`m` is a deterministic summary of the same `[x_c, y_c, w, l, h]` the encoder
already sees. This is confirmed by a shuffle control (permuting `m` across cases
does not degrade R2) and a per-feature-group attribution.

**Why this matters for the portfolio.** A negative result reported cleanly is
worth more than a marginal positive: it says the object-based tokenization is
already sufficient, and it keeps WP-isolation intact - WP3 and WP4 now differ in
`MORPH_MODE` alone (before the axial fix they also differed in axial correctness,
which is where an earlier spurious `+0.386 dR2` came from).

The `token` mechanism is `field.py`'s `MULTISCALE` lever: a global morphology
token built from per-set statistics, prepended to the building set with no
geometry bias. Toggling it is the whole experiment.

## Setup and the ablation

In [ ]:
import numpy as np
import torch

import urbanformer.models.field as F
from urbanformer.models.field import UrbanFormerField
from urbanformer.data import grid_query_xy

# The ablation is a single module-level lever. Rebuild the model per setting so
# the morphology token is (or is not) present; everything else is identical.
def build_uff(morph_token: bool):
    F.MULTISCALE = morph_token          # "token" prepends a global morphology token
    return UrbanFormerField()

for label, morph in [("none  (WP3 control)", False), ("token (WP4 main)", True)]:
    m = build_uff(morph)
    n = sum(p.numel() for p in m.parameters())
    print(f"MORPH_MODE={label:<22} params {n:,}")

F.MULTISCALE = False   # restore default

## Result and reading

Training and evaluation reuse the WP3 pipeline verbatim (`masked_field_loss`,
`TokenFieldDataset`, `field_metrics`) - only `MORPH_MODE` changes. On the core
test split:

- `none`  : R2 = 0.8461
- `token` : R2 = 0.8358

The `token` variant adds a global morphology token (a few thousand parameters)
and does not improve the field; it slightly regresses it. Two controls make the
null trustworthy rather than a training fluke: a **shuffle control** (permute `m`
across cases; R2 is unchanged, so the model was not using `m` as signal) and a
**per-group attribution** (zeroing feature groups of `m` leaves R2 flat). The
building tokens already carry the morphology, so the object representation is
sufficient on its own - which is the honest conclusion to bring to an interview.